# 05 · Objective success metric

A shot is **successful** if the machine delivered what the operators asked for: the actual signal is interpolated onto the target's timestamps, and **peak actual vs. peak target must agree within 15 %** — checked independently for plasma current and total NBI power, then combined (all available checks must pass).

**Worked example (shot 144229):** I<sub>p</sub> 1.20 MA target vs. 1.16 MA actual (3.2 %), NBI 2.24 MW vs. 2.20 MW (1.7 %) → success.

**Result:** 3,331 shots scored (89.6 % of those with parameters) → 2,426 hit target, 905 missed.

In [ ]:
import os, json, csv
import numpy as np
import pandas as pd

param_csv_path = "/path/to/outputs/shot_parameters_v2.csv"
targets_dir = "/path/to/outputs/shot_targets_json"

df = pd.read_csv(param_csv_path)
param_shots = set(df["shot_number"].tolist())
target_shots = set(int(f.replace(".json", "")) for f in os.listdir(targets_dir) if f.endswith(".json"))

eligible_shots = sorted(param_shots & target_shots)
print("Shots with parameters:", len(param_shots))
print("Shots with target JSON files:", len(target_shots))
print("Eligible for success check (both):", len(eligible_shots))

In [ ]:
def align_actual_to_target(target_time, target_data, actual_time, actual_data, actual_scale=1.0):
    """Interpolates actual onto target's (coarser, step-shaped) timestamps."""
    if not target_time or not actual_time:
        return None, None
    target_time, target_data = np.array(target_time), np.array(target_data)
    actual_time = np.array(actual_time)
    actual_data = np.array(actual_data) * actual_scale
    aligned_actual = np.interp(target_time, actual_time, actual_data)
    return target_data.tolist(), aligned_actual.tolist()


def score_target_vs_actual(series, actual_scale, use_abs, tolerance_pct):
    """
    Generic peak-based target-vs-actual score for one signal.
    use_abs: take abs() of values before comparing (handles sign-flipped
    target signals like IPTIPP without assuming direction)
    Returns dict with target/actual peaks, percent diff, and hit/miss.
    """
    t_time = series.get("target", {}).get("time")
    t_data = series.get("target", {}).get("data")
    a_time = series.get("actual", {}).get("time")
    a_data = series.get("actual", {}).get("data")

    if not (t_time and t_data and a_time and a_data):
        return None

    target, actual_aligned = align_actual_to_target(t_time, t_data, a_time, a_data, actual_scale)
    target_arr, actual_arr = np.array(target), np.array(actual_aligned)
    if use_abs:
        target_arr, actual_arr = np.abs(target_arr), np.abs(actual_arr)

    if len(target_arr) == 0:
        return None

    peak_target = float(np.max(target_arr))
    peak_actual = float(np.max(actual_arr)) if len(actual_arr) else None

    if not peak_target or peak_actual is None:
        return None

    pct_diff = abs(peak_actual - peak_target) / peak_target * 100
    return {
        "target_peak": peak_target,
        "actual_peak": peak_actual,
        "percent_diff": round(pct_diff, 1),
        "hit_target": pct_diff <= tolerance_pct,
    }


print("Alignment and scoring functions defined")

In [ ]:
def compute_shot_success(shot_number, targets_dir, tolerance_pct=15):
    """
    Objective success check: how closely actual IP and actual total NBI
    power tracked their real commanded targets during the shot.
    Only meaningful for shots with usable target JSON data - returns
    None fields if the underlying signal is missing.
    """
    path = os.path.join(targets_dir, f"{shot_number}.json")
    if not os.path.exists(path):
        return None
    with open(path, "r") as f:
        data = json.load(f)

    ip_result = score_target_vs_actual(data.get("ip", {}), actual_scale=1.0, use_abs=True, tolerance_pct=tolerance_pct)
    nbi_result = score_target_vs_actual(data.get("nbi_power", {}), actual_scale=1e-6, use_abs=False, tolerance_pct=tolerance_pct)

    if ip_result is None and nbi_result is None:
        return None

    row = {"shot_number": shot_number}
    row["ip_target_peak_A"] = ip_result["target_peak"] if ip_result else None
    row["ip_actual_peak_A"] = ip_result["actual_peak"] if ip_result else None
    row["ip_percent_diff"] = ip_result["percent_diff"] if ip_result else None
    row["ip_hit_target"] = ip_result["hit_target"] if ip_result else None

    row["nbi_target_peak_MW"] = nbi_result["target_peak"] if nbi_result else None
    row["nbi_actual_peak_MW"] = nbi_result["actual_peak"] if nbi_result else None
    row["nbi_percent_diff"] = nbi_result["percent_diff"] if nbi_result else None
    row["nbi_hit_target"] = nbi_result["hit_target"] if nbi_result else None

    known_hits = [v for v in (row["ip_hit_target"], row["nbi_hit_target"]) if v is not None]
    row["overall_success"] = all(known_hits) if known_hits else None

    return row


print("Success function defined")

In [ ]:
test_shot = 144229
result = compute_shot_success(test_shot, targets_dir)
print(result)

In [ ]:
success_rows = []
skipped = 0

for i, shot_number in enumerate(eligible_shots):
    if i % 500 == 0:
        print("Progress:", i, "/", len(eligible_shots))
    row = compute_shot_success(shot_number, targets_dir)
    if row is None:
        skipped += 1
        continue
    success_rows.append(row)

print("Scored:", len(success_rows), "| Skipped (no usable target data):", skipped)

output_file = "/path/to/outputs/shot_success_scores.csv"
fieldnames = ["shot_number", "ip_target_peak_A", "ip_actual_peak_A", "ip_percent_diff", "ip_hit_target",
              "nbi_target_peak_MW", "nbi_actual_peak_MW", "nbi_percent_diff", "nbi_hit_target", "overall_success"]
with open(output_file, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(success_rows)

print("Saved to:", output_file)

overall_true = sum(1 for r in success_rows if r["overall_success"] is True)
overall_false = sum(1 for r in success_rows if r["overall_success"] is False)
print(f"Overall success: {overall_true} | Not successful: {overall_false}")

In [ ]:
import os
targets_dir = "/path/to/outputs/shot_targets_json"
json_files = [f for f in os.listdir(targets_dir) if f.endswith(".json")]
print("Actual JSON files on disk:", len(json_files))